# Build FAISS Index

This notebook demonstrates building a FAISS index for job matching.

In [ ]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
# Load sample jobs
with open('../data/jobs/jobs.json', 'r') as f:
    jobs = json.load(f)

print(f"Loaded {len(jobs)} jobs")
print(f"Sample job: {jobs[0]['title']}")

In [ ]:
# Initialize embedding model
model = SentenceTransformer('BAAI/bge-small-en-v1.5')

# Create text representations of jobs
job_texts = [
    f"{job['title']} {job['skills']} {job['description']}"
    for job in jobs
]

# Generate embeddings
embeddings = model.encode(job_texts, normalize_embeddings=True)
print(f"Generated embeddings shape: {embeddings.shape}")

In [ ]:
# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity with normalized vectors
index.add(embeddings.astype(np.float32))

print(f"FAISS index built with {index.ntotal} vectors")

In [ ]:
# Test search
query = "Python developer with machine learning skills"
query_embedding = model.encode([query], normalize_embeddings=True)

# Search
k = 5
scores, indices = index.search(query_embedding.astype(np.float32), k)

print(f"\nQuery: {query}")
print(f"\nTop {k} matches:")
for score, idx in zip(scores[0], indices[0]):
    print(f"  {jobs[idx]['title']} - Score: {score:.4f}")

In [ ]:
# Save the index
faiss.write_index(index, '../vectorstore/jobs.index')
print("Index saved to ../vectorstore/jobs.index")